Algoritmo para gerar planos de corte para peças retas (MDF liso):

In [1]:
!pip install rectpack

  Preparing metadata (setup.py) ... done
  Created wheel for rectpack: filename=rectpack-0.2.2-py3-none-any.whl size=19374 sha256=82725da6506da8de08b5fb64f96fc74f7817a06120bec5579c5478e5990230b1
  Stored in directory: /root/.cache/pip/wheels/a0/39/c6/e088f7d582ebef64f5cc9138e5455e7fd86c48ac9dc51bc892
Successfully built rectpack


In [2]:
import html
from collections import Counter

import pandas as pd
import rectpack as rp

LARGURA_CHAPA = 1850
ALTURA_CHAPA = 2750
ESPACAMENTO = 15

In [3]:
df = pd.read_excel("/content/lista_medidas.xlsx")

In [4]:
print(df.head())
print(df.info())
print(df.columns)

     nome  comprimento  largura  qtd  espessura
0  peca 1          200      800    2          6
1  peca 2          400      600    4         15
2  peca 3          600      100    3         18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   nome         3 non-null      object
 1   comprimento  3 non-null      int64 
 2   largura      3 non-null      int64 
 3   qtd          3 non-null      int64 
 4   espessura    3 non-null      int64 
dtypes: int64(4), object(1)
memory usage: 252.0+ bytes
None
Index(['nome', 'comprimento', 'largura', 'qtd', 'espessura'], dtype='object')


In [5]:
colunas_necessarias = ['nome', 'comprimento', 'largura', 'qtd', 'espessura']

faltando = set(colunas_necessarias) - set(df.columns)
if faltando:
    raise ValueError(f'Colunas ausentes na planilha: {faltando}')

linhas_vazias = df[df[colunas_necessarias].isna().any(axis=1)]
if not linhas_vazias.empty:
    raise ValueError(
        f'Há células vazias na planilha. Verifique as linhas:\n{linhas_vazias}'
    )

print('Planilha validada.')

Planilha validada.


In [6]:
pecas = []

for _, linha in df.iterrows():
    for i in range(int(linha['qtd'])):
        pecas.append({
            'nome': linha['nome'],
            'comprimento': linha['comprimento'],
            'largura': linha['largura'],
            'espessura': linha['espessura']
        })

print(pecas)

[{'nome': 'peca 1', 'comprimento': 200, 'largura': 800, 'espessura': 6}, {'nome': 'peca 1', 'comprimento': 200, 'largura': 800, 'espessura': 6}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}]


In [7]:
print(f'Total de peças: {len(pecas)}')
print(f'Quantidade por nome: {Counter(p["nome"] for p in pecas)}')
print(f'Lista de peças: {pecas}')

Total de peças: 9
Quantidade por nome: Counter({'peca 2': 4, 'peca 3': 3, 'peca 1': 2})
Lista de peças: [{'nome': 'peca 1', 'comprimento': 200, 'largura': 800, 'espessura': 6}, {'nome': 'peca 1', 'comprimento': 200, 'largura': 800, 'espessura': 6}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 2', 'comprimento': 400, 'largura': 600, 'espessura': 15}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}, {'nome': 'peca 3', 'comprimento': 600, 'largura': 100, 'espessura': 18}]


In [8]:
resultado = []
espessura_da_chapa = {}
deslocamento = 0

for espessura in sorted({p['espessura'] for p in pecas}):
    packer = rp.newPacker(
        rotation=True,
        pack_algo=rp.MaxRectsBssf,
        bin_algo=rp.PackingBin.BBF,
        sort_algo=rp.SORT_AREA
    )
    packer.add_bin(LARGURA_CHAPA, ALTURA_CHAPA, count=float("inf"))

    for i, peca in enumerate(pecas):
        if peca['espessura'] == espessura:
            packer.add_rect(
                width=peca['largura'] + ESPACAMENTO,
                height=peca['comprimento'] + ESPACAMENTO,
                rid=i
            )

    packer.pack()
    parcial = packer.rect_list()

    for bin_id, x, y, w, h, rid in parcial:
        resultado.append((bin_id + deslocamento, x, y, w, h, rid))
        espessura_da_chapa[bin_id + deslocamento] = espessura

    deslocamento += len({item[0] for item in parcial})

for bin_id, x, y, w, h, rid in resultado:
    peca = pecas[rid]

    print(
        f'Chapa: {bin_id + 1} ({espessura_da_chapa[bin_id]} mm) | '
        f'Peça: {peca["nome"]} | '
        f'Posição reservada: ({x}, {y}) | '
        f'Área reservada: {w} x {h} mm | '
        f'Dimensões originais: '
        f'{peca["largura"]} x {peca["comprimento"]} mm'
    )

Chapa: 1 (6 mm) | Peça: peca 1 | Posição reservada: (0, 0) | Área reservada: 815 x 215 mm | Dimensões originais: 800 x 200 mm
Chapa: 1 (6 mm) | Peça: peca 1 | Posição reservada: (815, 0) | Área reservada: 815 x 215 mm | Dimensões originais: 800 x 200 mm
Chapa: 2 (15 mm) | Peça: peca 2 | Posição reservada: (0, 0) | Área reservada: 615 x 415 mm | Dimensões originais: 600 x 400 mm
Chapa: 2 (15 mm) | Peça: peca 2 | Posição reservada: (615, 0) | Área reservada: 615 x 415 mm | Dimensões originais: 600 x 400 mm
Chapa: 2 (15 mm) | Peça: peca 2 | Posição reservada: (1230, 0) | Área reservada: 615 x 415 mm | Dimensões originais: 600 x 400 mm
Chapa: 2 (15 mm) | Peça: peca 2 | Posição reservada: (0, 415) | Área reservada: 615 x 415 mm | Dimensões originais: 600 x 400 mm
Chapa: 3 (18 mm) | Peça: peca 3 | Posição reservada: (0, 0) | Área reservada: 615 x 115 mm | Dimensões originais: 100 x 600 mm
Chapa: 3 (18 mm) | Peça: peca 3 | Posição reservada: (615, 0) | Área reservada: 615 x 115 mm | Dimensões

In [9]:
ids_alocados = {item[5] for item in resultado}

ids_nao_alocados = set(range(len(pecas))) - ids_alocados

print(f'Total solicitado: {len(pecas)}')
print(f'Total encaixado: {len(ids_alocados)}')
print(f'Total não encaixado: {len(ids_nao_alocados)}')

for rid in sorted(ids_nao_alocados):
    print(f'Não encaixada: {pecas[rid]["nome"]}')

Total solicitado: 9
Total encaixado: 9
Total não encaixado: 0


In [10]:
area_chapa = LARGURA_CHAPA * ALTURA_CHAPA
area_pecas = sum(
    largura * altura
    for _, x, y, largura, altura, rid in resultado
)

chapas_utilizadas = {
    item[0] for item in resultado
}

area_total_chapas = len(chapas_utilizadas) * area_chapa

if area_total_chapas:
    aproveitamento = area_pecas / area_total_chapas * 100
    print(f'Chapas utilizadas: {len(chapas_utilizadas)}')
    print(f'Aproveitamento: {aproveitamento:.2f}%')

Chapas utilizadas: 3
Aproveitamento: 10.38%


In [11]:
CHAPAS_POR_LINHA = 3
FOLGA_HORIZONTAL = 100
FOLGA_VERTICAL = 100
MARGEM_TOPO = 120

ids_chapas = sorted({item[0] for item in resultado})
qtd_chapas = len(ids_chapas)

colunas = min(CHAPAS_POR_LINHA, qtd_chapas)
linhas = -(-qtd_chapas // CHAPAS_POR_LINHA)

altura_bloco = MARGEM_TOPO + ALTURA_CHAPA + FOLGA_VERTICAL

largura_total = colunas * LARGURA_CHAPA + (colunas - 1) * FOLGA_HORIZONTAL
altura_total = linhas * altura_bloco - FOLGA_VERTICAL


def origem_chapa(posicao):
    coluna = posicao % CHAPAS_POR_LINHA
    linha = posicao // CHAPAS_POR_LINHA
    x0 = coluna * (LARGURA_CHAPA + FOLGA_HORIZONTAL)
    y0 = linha * altura_bloco + MARGEM_TOPO
    return x0, y0


elementos = [
    f'<svg xmlns="http://www.w3.org/2000/svg" '
    f'width="{largura_total}mm" '
    f'height="{altura_total}mm" '
    f'viewBox="0 0 {largura_total} {altura_total}">'
]

for posicao, bin_id in enumerate(ids_chapas):
    x0, y0 = origem_chapa(posicao)

    espessura = espessura_da_chapa.get(bin_id)
    titulo = f'Chapa {bin_id + 1}'
    if espessura is not None:
        titulo += f' - {espessura} mm'

    elementos.append(
        f'<text x="{x0 + LARGURA_CHAPA / 2}" y="{y0 - MARGEM_TOPO / 2}" '
        f'font-size="50" font-weight="bold" text-anchor="middle" '
        f'dominant-baseline="middle" fill="black">'
        f'{html.escape(titulo)}</text>'
    )

    elementos.append(
        f'<rect x="{x0}" y="{y0}" '
        f'width="{LARGURA_CHAPA}" height="{ALTURA_CHAPA}" '
        f'fill="white" stroke="black" stroke-width="2"/>'
    )

for bin_id, x, y, w, h, rid in resultado:
    peca = pecas[rid]

    largura_original = peca["largura"]
    comprimento_original = peca["comprimento"]

    if (
        abs(w - (largura_original + ESPACAMENTO)) < 0.001
        and abs(h - (comprimento_original + ESPACAMENTO)) < 0.001
    ):
        largura_real = largura_original
        altura_real = comprimento_original

    elif (
        abs(w - (comprimento_original + ESPACAMENTO)) < 0.001
        and abs(h - (largura_original + ESPACAMENTO)) < 0.001
    ):
        largura_real = comprimento_original
        altura_real = largura_original

    else:
        raise ValueError(
            f'Não foi possível determinar a orientação da peça '
            f'{peca["nome"]}'
        )

    x0, y0 = origem_chapa(ids_chapas.index(bin_id))

    x_real = x0 + x + (w - largura_real) / 2
    y_real = y + (h - altura_real) / 2

    y_svg = y0 + ALTURA_CHAPA - y_real - altura_real

    nome = html.escape(str(peca["nome"]))
    medidas = f'{largura_real} x {altura_real}'

    elementos.append(
        f'<rect x="{x_real}" y="{y_svg}" '
        f'width="{largura_real}" height="{altura_real}" '
        f'fill="none" stroke="black" stroke-width="1"/>'
    )

    centro_x = x_real + largura_real / 2
    centro_y = y_svg + altura_real / 2

    girar = largura_real < 200 and altura_real > largura_real
    rotacao = f' transform="rotate(-90 {centro_x} {centro_y})"' if girar else ''

    elementos.append(
        f'<text x="{centro_x}" y="{centro_y}" '
        f'font-size="25" text-anchor="middle" '
        f'dominant-baseline="middle" fill="black"{rotacao}>'
        f'<tspan x="{centro_x}" dy="-0.6em">{nome}</tspan>'
        f'<tspan x="{centro_x}" dy="1.2em">{medidas}</tspan>'
        f'</text>'
    )

elementos.append("</svg>")

with open("plano_corte.svg", "w", encoding="utf-8") as arquivo:
    arquivo.write("\n".join(elementos))

print("SVG gerado: plano_corte.svg")

SVG gerado: plano_corte.svg
